# NxM validation: held-out detector pulses

This notebook applies the regularized 12-channel fit to a raw series not retained in the saved PCA training sample. The ignored `data` symlink is required to reproduce the recorded results.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import hashlib
import os
from pathlib import Path
import sys

import joblib
import matplotlib.pyplot as plt
import numpy as np

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / 'src'))
from nxm import NxMReconstructor

paths = {
    'templates': REPO_ROOT / 'data/pca/templates_pca_543.joblib',
    'csd': REPO_ROOT / 'data/final_psd_and_templates/csd_results.joblib',
    'training': REPO_ROOT / 'data/pca/traces_pca_543.joblib',
    'candidate': REPO_ROOT / 'data/trace_signal/trace_z1_23231216_233807.joblib',
}
DATA_AVAILABLE = all(path.exists() for path in paths.values())
if not DATA_AVAILABLE:
    missing = [str(path) for path in paths.values() if not path.exists()]
    print('Held-out validation skipped. Restore the ignored data symlink and these artifacts:')
    print('\n'.join(missing))

## Load artifacts and prove the holdout split

In [ ]:
if DATA_AVAILABLE:
    templates = joblib.load(paths['templates'], mmap_mode='r')
    csd_artifact = joblib.load(paths['csd'], mmap_mode='r')
    training = joblib.load(paths['training'], mmap_mode='r')
    candidate = joblib.load(paths['candidate'], mmap_mode='r')
    channels = list(csd_artifact['channels_ordered'])

    def fingerprint(trace):
        return hashlib.blake2b(np.asarray(trace).tobytes(), digest_size=16).digest()

    training_ids = {fingerprint(trace) for trace in training[channels[0]]}
    overlap = [index for index, trace in enumerate(candidate[channels[0]]) if fingerprint(trace) in training_ids]
    held_out_indices = [index for index in range(len(candidate[channels[0]])) if index not in overlap]
    print(f'Candidate events: {len(candidate[channels[0]])}')
    print(f'PCA-training overlap: {overlap}')
    print(f'Held-out events: {len(held_out_indices)}')
    assert overlap == [9]
    assert len(held_out_indices) == 26

## Fit all held-out events

The fit uses 4,000 low-frequency RFFT bins, the full 12-channel CSD, a ±150-sample initialization search, and a ±10-sample final window. Independent events are evaluated concurrently; the fit itself remains deterministic.

In [ ]:
if DATA_AVAILABLE:
    means = np.stack([templates['template_avg'][channel] for channel in channels])
    components = np.stack([templates['template_pca'][channel] for channel in channels])
    variances = np.stack([templates['variance_pca'][channel] for channel in channels])
    csd = np.asarray(csd_artifact['csd_matrices'])
    reconstructor = NxMReconstructor(
        means, components, variances, csd,
        channel_names=channels, frequency_limit=4000,
    )
    mean_only = NxMReconstructor(
        means, np.empty((len(channels), 0, means.shape[1])),
        np.empty((len(channels), 0)), csd,
        channel_names=channels, frequency_limit=4000,
    )
    traces = []
    for event_index in held_out_indices:
        event = np.stack([np.asarray(candidate[channel][event_index], dtype=float) for channel in channels])
        event -= np.mean(event[:, :16000], axis=1, keepdims=True)
        traces.append(event)

    def fit_event(event):
        amplitudes, delays = reconstructor.initialize(event, (-150, 150))
        return reconstructor.fit(
            event, initial_amplitudes=amplitudes, initial_delays=delays,
            delay_half_width=10,
            optimizer_options={'xtol': 0.1, 'ftol': 1e-2, 'maxiter': 20, 'maxfev': 1400},
        )

    workers = min(6, os.cpu_count() or 1)
    with ThreadPoolExecutor(max_workers=workers) as executor:
        fit_results = list(executor.map(fit_event, traces))
    print(f'Completed {len(fit_results)} fits with {workers} workers.')

## Fit and residual diagnostics

In [ ]:
if DATA_AVAILABLE:
    amplitudes = np.stack([result.amplitudes for result in fit_results])
    delays = np.stack([result.delays for result in fit_results])
    coefficients = np.stack([result.shape_coefficients for result in fit_results])
    objectives = np.array([result.objective for result in fit_results])
    mean_only_objectives = np.array([
        mean_only.profile(event, result.amplitudes, result.delays).objective
        for event, result in zip(traces, fit_results)
    ])
    improvements = mean_only_objectives - objectives
    converged = np.array([result.success for result in fit_results])
    amplitude_bounds = np.array([np.sum(result.amplitude_at_bound) for result in fit_results])
    delay_bounds = np.array([np.sum(result.delay_at_bound) for result in fit_results])
    reduced_residual = np.array([result.residual_objective for result in fit_results]) / (len(channels) * (2 * 3999))
    print(f'Optimizer convergence: {np.mean(converged):.1%}')
    print(f'Median posterior improvement over z=0: {np.median(improvements):.1f}')
    print(f'Median reduced residual objective: {np.median(reduced_residual):.3f}')
    print(f'Events with an amplitude at zero: {np.sum(amplitude_bounds > 0)}/{len(fit_results)}')
    print(f'Events with a delay at its bound: {np.sum(delay_bounds > 0)}/{len(fit_results)}')
    print(f'Largest absolute PCA pull: {np.max(np.abs(coefficients)):.2f}')
    assert np.all(np.isfinite(objectives))
    assert np.all(improvements >= -1e-6 * np.maximum(1, mean_only_objectives))

### Interpretation

The finite objectives, 100% optimizer convergence, and positive posterior improvement validate the numerical workflow on unseen waveforms. The frequent timing-bound hits and the largest PCA pull of 9.49 are also important: this unfiltered series contains channels or events outside the current timing window and PCA training distribution. These fits should remain diagnostic, not production reconstruction, until event-quality selection, relative-delay calibration, and the PCA training population are revisited.

In [ ]:
if DATA_AVAILABLE:
    fig, axes = plt.subplots(2, 2, figsize=(11, 7))
    axes[0, 0].hist(improvements, bins=15); axes[0, 0].set_title('Posterior improvement over z=0')
    axes[0, 1].hist(coefficients.ravel(), bins=25); axes[0, 1].set_title('Standardized PCA coefficients')
    axes[1, 0].hist(delays.ravel(), bins=25); axes[1, 0].set_title('Fitted delay shifts [samples]')
    axes[1, 1].hist(reduced_residual, bins=15); axes[1, 1].set_title('Reduced residual objective')
    plt.tight_layout(); plt.show()

## Representative reconstruction

In [ ]:
if DATA_AVAILABLE:
    representative = int(np.argmax(improvements))
    model = reconstructor.reconstruct(fit_results[representative])
    selected_channels = [0, 1, 6, 7]
    fig, axes = plt.subplots(2, 2, figsize=(12, 6), sharex=True)
    for axis, channel_index in zip(axes.ravel(), selected_channels):
        window = slice(15850, 16750)
        axis.plot(np.arange(means.shape[1])[window], traces[representative][channel_index, window], label='held-out data', alpha=0.7)
        axis.plot(np.arange(means.shape[1])[window], model[channel_index, window], label='NxM model')
        axis.plot(np.arange(means.shape[1])[window], (traces[representative] - model)[channel_index, window], label='residual', alpha=0.7)
        axis.set_title(channels[channel_index]); axis.legend(fontsize=8)
    fig.suptitle(f'Held-out event {held_out_indices[representative]}')
    plt.tight_layout(); plt.show()